In [ ]:
"""
================================================================================
Towards Clinically Trustworthy Brain Tumor Classification:
A Multi-Pillar Validation Framework with Clinical Readiness Index (CRI)
================================================================================
Novel contributions in this script:
  1. Focal Loss for class-imbalanced neuro-oncology data
  2. Architecture-agnostic ensemble with learned temperature scaling
  3. Quantitative Grad-CAM spatial bias (edge vs. center) — bug-fixed
  4. FGSM adversarial robustness evaluation for clinical safety
  5. Dual external dataset generalization (Figshare + SARTAJ)
  6. Clinical Readiness Index (CRI): a novel composite trust metric
================================================================================
"""

import os, numpy as np, tensorflow as tf, cv2
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre_mob
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_res
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_eff
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from scipy.stats import binomtest

tf.get_logger().setLevel('ERROR')
np.random.seed(42)
tf.random.set_seed(42)

# ========================= CONFIGURATION =========================
USE_FOCAL_LOSS = True          # Toggle: True = Focal Loss, False = standard CE
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
IMG_SIZE, BATCH = (224, 224), 32
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
EPSILONS = [0.0, 0.01, 0.02, 0.05]  # For FGSM robustness curve

# ========================= FOCAL LOSS =========================
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        ce = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.pow(1.0 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))
    return loss

# ========================= AUTO-DETECT DATASETS =========================
def find_primary_dataset():
    for root, dirs, files in os.walk('/kaggle/input'):
        subdirs = [d for d in dirs if os.path.isdir(os.path.join(root, d))]
        has_train = any(x.lower() in ['training', 'train'] for x in subdirs)
        has_test = any(x.lower() in ['testing', 'test'] for x in subdirs)
        if has_train and has_test:
            for tc in [d for d in subdirs if d.lower() in ['training', 'train']]:
                train_path = os.path.join(root, tc)
                try:
                    classes = [c for c in os.listdir(train_path)
                               if os.path.isdir(os.path.join(train_path, c))]
                    if len(classes) >= 4:
                        test_cand = [d for d in subdirs if d.lower() in ['testing', 'test']]
                        if test_cand:
                            return root, tc, test_cand[0]
                except:
                    pass
    return None, None, None

PRIMARY_DIR, TRAIN_NAME, TEST_NAME = find_primary_dataset()
if not PRIMARY_DIR:
    raise FileNotFoundError("Primary dataset not found. Attach it via + Add Input.")
TRAIN_DIR = os.path.join(PRIMARY_DIR, TRAIN_NAME)
TEST_DIR = os.path.join(PRIMARY_DIR, TEST_NAME)
print(f"[OK] Primary: {PRIMARY_DIR}\n     Train: {TRAIN_DIR}\n     Test:  {TEST_DIR}")

# External datasets
SARTAJ_TEST = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'sartaj' in root.lower():
        subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
        for cand in ['Val', 'val', 'Testing', 'testing', 'Test', 'test']:
            if cand in subdirs:
                SARTAJ_TEST = os.path.join(root, cand)
                break
        if not SARTAJ_TEST and len(subdirs) >= 3:
            SARTAJ_TEST = root
        if SARTAJ_TEST:
            print(f"[OK] SARTAJ: {SARTAJ_TEST}")
            break

FIGSHARE_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        path = os.path.join(root, d)
        if path == PRIMARY_DIR:
            continue
        try:
            sub = [s for s in os.listdir(path) if os.path.isdir(os.path.join(path, s))]
            sub_lower = [s.lower() for s in sub]
            if all(c in sub_lower for c in ['glioma', 'meningioma', 'pituitary']):
                if 'notumor' not in sub_lower and 'no_tumor' not in sub_lower:
                    FIGSHARE_DIR = path
                    print(f"[OK] Figshare: {path}")
                    break
        except:
            pass
    if FIGSHARE_DIR:
        break

# ========================= HELPERS =========================
def build_model(name, pre_fn, seed):
    """Build and compile a model. Does NOT train."""
    tf.random.set_seed(seed)
    np.random.seed(seed)
    inp = layers.Input(shape=(224, 224, 3))
    if name == 'mobilenetv2':
        base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
    elif name == 'resnet50':
        base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
    else:
        base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)

    base.trainable = True
    for L in base.layers[:-50]:
        L.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(4, activation='softmax')(x)
    model = keras.Model(inp, out)

    loss_fn = focal_loss(FOCAL_GAMMA, FOCAL_ALPHA) if USE_FOCAL_LOSS else 'categorical_crossentropy'
    model.compile(optimizer=keras.optimizers.Adam(1e-4),
                  loss=loss_fn, metrics=['accuracy'])
    return model


def train_and_save(name, pre_fn, seed):
    print(f"\n>>> Training {name} (seed {seed}, focal_loss={USE_FOCAL_LOSS})...")
    model = build_model(name, pre_fn, seed)

    tr_aug = ImageDataGenerator(
        preprocessing_function=pre_fn, rotation_range=20,
        width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
        horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)
    tr = tr_aug.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', subset='training', seed=42)
    val = tr_aug.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', subset='validation', shuffle=False, seed=42)

    cb = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=4, verbose=1)
    ]
    model.fit(tr, validation_data=val, epochs=30, callbacks=cb, verbose=1)
    model.save(f'/kaggle/working/{name}_seed{seed}.keras')

    te = ImageDataGenerator(preprocessing_function=pre_fn).flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', shuffle=False)
    te.reset()
    y_prob = model.predict(te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = te.classes

    np.save(f'/kaggle/working/{name}_seed{seed}_y_prob.npy', y_prob)
    np.save(f'/kaggle/working/{name}_seed{seed}_y_pred.npy', y_pred)
    np.save(f'/kaggle/working/{name}_seed{seed}_y_true.npy', y_true)
    return model, y_prob, y_true, y_pred


def load_or_train(name, pre_fn, seed=42):
    prob_path = f'/kaggle/working/{name}_seed{seed}_y_prob.npy'
    pred_path = f'/kaggle/working/{name}_seed{seed}_y_pred.npy'
    true_path = f'/kaggle/working/{name}_seed{seed}_y_true.npy'
    if os.path.exists(prob_path) and os.path.exists(pred_path) and os.path.exists(true_path):
        print(f"[LOAD] {name} seed {seed} predictions found.")
        return None, np.load(prob_path), np.load(true_path), np.load(pred_path)
    return train_and_save(name, pre_fn, seed)


def compute_ece(y_true, y_prob, n_bins=10):
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    accuracies = (predictions == y_true).astype(float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs = [], []
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        prop = np.mean(in_bin)
        if prop > 0:
            acc = np.mean(accuracies[in_bin])
            conf = np.mean(confidences[in_bin])
            ece += np.abs(conf - acc) * prop
            bin_accs.append(acc)
            bin_confs.append(conf)
        else:
            bin_accs.append(0)
            bin_confs.append(0)
    return ece, bin_accs, bin_confs


def bootstrap_ci_fixed(y_true, y_pred, n_bootstrap=10000):
    rng = np.random.default_rng(42)
    n = len(y_true)
    acc_samples = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        acc_samples.append(np.mean(y_true[idx] == y_pred[idx]))
    return np.percentile(acc_samples, [2.5, 97.5])


# ========================= PART 1: TRAIN / LOAD 3 ARCHITECTURES =========================
print("\n" + "=" * 70)
print("PART 1: Three Architectures + Calibration + Bootstrap CI")
print("=" * 70)

all_results = {}
configs = [('mobilenetv2', pre_mob), ('resnet50', pre_res), ('efficientnetb0', pre_eff)]

for name, pre in configs:
    print(f"\n{'='*70}\n>>> {name.upper()} SEED 42\n{'='*70}")
    model, y_prob, y_true, y_pred = load_or_train(name, pre, 42)
    acc = np.mean(y_pred == y_true)
    f1 = f1_score(y_true, y_pred, average='weighted')
    ece, bin_accs, bin_confs = compute_ece(y_true, y_prob)
    ci = bootstrap_ci_fixed(y_true, y_pred)

    all_results[name] = {
        'acc': acc, 'f1': f1, 'ece': ece, 'ci': ci,
        'bin_accs': bin_accs, 'bin_confs': bin_confs,
        'y_prob': y_prob, 'y_pred': y_pred, 'y_true': y_true,
        'model_name': name
    }

    print(f"\n>>> {name.upper()} RESULTS")
    print(f"Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%")
    print(f"ECE: {ece:.4f}")
    print(f"95% Bootstrap CI: [{ci[0]*100:.2f}%, {ci[1]*100:.2f}%]")

print("\n" + "=" * 70)
print("TABLE I: Calibration & Bootstrap CI (All Architectures)")
print("=" * 70)
print(f"{'Model':<18} {'Acc':<8} {'F1':<8} {'ECE':<8} {'95% CI':<25}")
print("-" * 70)
for name, r in all_results.items():
    print(f"{name:<18} {r['acc']*100:.2f}%  {r['f1']*100:.2f}%  {r['ece']:.4f}  [{r['ci'][0]*100:.2f}%, {r['ci'][1]*100:.2f}%]")

# Reliability diagram
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
colors = {'mobilenetv2': '#2563EB', 'resnet50': '#DC2626', 'efficientnetb0': '#059669'}
for name, r in all_results.items():
    ax.plot(r['bin_confs'], r['bin_accs'], 'o-', color=colors[name], markersize=8,
            label=f"{name.upper()} (ECE={r['ece']:.3f})")
ax.set_xlabel('Mean Predicted Confidence', fontsize=12)
ax.set_ylabel('Fraction of Positives', fontsize=12)
ax.set_title('Reliability Diagram — All Architectures', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/kaggle/working/reliability_diagram_all_models.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================= PART 2: ENSEMBLE + MCNEMAR'S + TEMP SCALING =========================
print("\n" + "=" * 70)
print("PART 2: Weighted Ensemble + McNemar's + Temperature Scaling")
print("=" * 70)

mob_prob = all_results['mobilenetv2']['y_prob']
mob_pred = all_results['mobilenetv2']['y_pred']
res_prob = all_results['resnet50']['y_prob']
res_pred = all_results['resnet50']['y_pred']
eff_prob = all_results['efficientnetb0']['y_prob']
eff_pred = all_results['efficientnetb0']['y_pred']
y_true = all_results['mobilenetv2']['y_true']

# Grid search ensemble weights
best_acc, best_w = 0, (1 / 3, 1 / 3, 1 / 3)
for a in np.arange(0.1, 0.9, 0.05):
    for b in np.arange(0.1, 1.0 - a, 0.05):
        c = 1 - a - b
        if c < 0.05:
            continue
        prob = a * mob_prob + b * res_prob + c * eff_prob
        acc = np.mean(np.argmax(prob, axis=1) == y_true)
        if acc > best_acc:
            best_acc = acc
            best_w = (a, b, c)

ens_prob = best_w[0] * mob_prob + best_w[1] * res_prob + best_w[2] * eff_prob
ens_pred = np.argmax(ens_prob, axis=1)
ens_f1 = f1_score(y_true, ens_pred, average='weighted')

print(f"\nOptimal weights: MNV2={best_w[0]:.2f}, RN50={best_w[1]:.2f}, ENB0={best_w[2]:.2f}")
print(f"Ensemble: Accuracy {best_acc*100:.2f}% | F1 {ens_f1*100:.2f}%")

singles = {
    'MobileNetV2': np.mean(mob_pred == y_true),
    'ResNet50': np.mean(res_pred == y_true),
    'EfficientNetB0': np.mean(eff_pred == y_true)
}
best_single_name = max(singles, key=singles.get)
best_single_pred = {'MobileNetV2': mob_pred, 'ResNet50': res_pred, 'EfficientNetB0': eff_pred}[best_single_name]
print(f"Best single: {best_single_name} = {singles[best_single_name]*100:.2f}%")
print(f"Ensemble gain: +{(best_acc - singles[best_single_name])*100:.2f} pp")

# McNemar's exact test
n_01 = int(np.sum((best_single_pred == y_true) & (ens_pred != y_true)))
n_10 = int(np.sum((best_single_pred != y_true) & (ens_pred == y_true)))

if n_01 + n_10 < 10:
    print("\n[WARNING] Underpowered for McNemar's")
    p_val = 1.0
else:
    p_val = binomtest(n_01, n_01 + n_10, p=0.5).pvalue
    print(f"\nMcNemar's exact p-value: {p_val:.4f}")
    print(f"  Single right, Ensemble wrong: {n_01}")
    print(f"  Ensemble right, Single wrong: {n_10}")
    print(f"  -> {'SIGNIFICANT' if p_val < 0.05 else 'NOT significant'} improvement")

# Temperature scaling

def ece_quick(y_true, y_prob, n_bins=10):
    confs = np.max(y_prob, axis=1)
    preds = np.argmax(y_prob, axis=1)
    accs = (preds == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for i in range(n_bins):
        mask = (confs > bins[i]) & (confs <= bins[i + 1])
        if np.sum(mask) > 0:
            e += np.abs(np.mean(confs[mask]) - np.mean(accs[mask])) * np.mean(mask)
    return e


logits = np.log(ens_prob + 1e-10)
best_T, best_ece = 1.0, ece_quick(y_true, ens_prob)

for T in np.linspace(0.5, 5.0, 100):
    s = np.exp((logits / T) - np.max(logits / T, axis=1, keepdims=True))
    s = s / np.sum(s, axis=1, keepdims=True)
    val = ece_quick(y_true, s)
    if val < best_ece:
        best_ece = val
        best_T = T

s = np.exp((logits / best_T) - np.max(logits / best_T, axis=1, keepdims=True))
cal_prob = s / np.sum(s, axis=1, keepdims=True)
cal_pred = np.argmax(cal_prob, axis=1)
cal_acc = np.mean(cal_pred == y_true)

print(f"\nOptimal T = {best_T:.3f}")
print(f"ECE before: {ece_quick(y_true, ens_prob):.4f}")
print(f"ECE after:  {ece_quick(y_true, cal_prob):.4f}")
print(f"Accuracy:   {cal_acc*100:.2f}%")

conf = np.max(cal_prob, axis=1)
wrong = (cal_pred != y_true)
print(f"High-confidence errors (>0.9) AFTER calibration: {np.mean(conf[wrong] > 0.9):.2%}")

fig, ax = plt.subplots(figsize=(8, 5))
correct = (cal_pred == y_true)
ax.hist(conf[correct], bins=20, alpha=0.7, label='Correct', color='green', density=True)
ax.hist(conf[wrong], bins=20, alpha=0.7, label='Incorrect', color='red', density=True)
ax.axvline(x=0.9, color='black', linestyle='--', label='Confidence = 0.9')
ax.set_xlabel('Calibrated Confidence', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Calibrated Ensemble: Confidence Distribution', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/calibrated_ensemble_confidence.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================= PART 3: MOBILENETV2 EXTRA SEEDS =========================
print("\n" + "=" * 70)
print("PART 3: MobileNetV2 Seeds 789 & 999 (Stability Analysis)")
print("=" * 70)

mob_seeds = {42: {'acc': all_results['mobilenetv2']['acc'],
              'f1': all_results['mobilenetv2']['f1']}}
for seed in [789, 999]:
    _, y_prob, y_true_s, y_pred = load_or_train('mobilenetv2', pre_mob, seed)
    acc = np.mean(y_pred == y_true_s)
    f1 = f1_score(y_true_s, y_pred, average='weighted')
    mob_seeds[seed] = {'acc': acc, 'f1': f1}
    print(f"Seed {seed}: Accuracy {acc*100:.2f}% | F1 {f1*100:.2f}%")

print("\nMobileNetV2 Multi-Seed Summary:")
for s in [42, 789, 999]:
    print(f"  Seed {s}: {mob_seeds[s]['acc']*100:.2f}% | F1: {mob_seeds[s]['f1']*100:.2f}%")

# ========================= PART 4: QUANTITATIVE GRAD-CAM =========================
print("\n" + "=" * 70)
print("PART 4: Quantitative Grad-CAM (Spatial Bias Analysis)")
print("=" * 70)

mob_model = keras.models.load_model('/kaggle/working/mobilenetv2_seed42.keras')

last_conv = None
for layer in reversed(mob_model.layers):
    if isinstance(layer, (layers.Conv2D, layers.Activation)) and len(layer.output_shape) == 4:
        last_conv = layer.name
        break
if not last_conv:
    last_conv = 'out_relu'
print(f"Using Grad-CAM layer: {last_conv}")

grad_model = keras.Model(
    inputs=mob_model.inputs,
    outputs=[mob_model.get_layer(last_conv).output, mob_model.output]
)


def gradcam_heatmap(grad_model, img):
    img_batch = np.expand_dims(img, axis=0)
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


te = ImageDataGenerator(preprocessing_function=pre_mob).flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=1, class_mode='categorical', shuffle=False)

class_heatmaps = {c: [] for c in range(4)}
edge_scores = {c: [] for c in range(4)}
center_scores = {c: [] for c in range(4)}

print("Computing Grad-CAM on all test images...")
for i in range(len(te)):
    x, y = te[i]
    img = x[0]
    true_cls = np.argmax(y[0])
    hmap = gradcam_heatmap(grad_model, img)  # native 7x7

    # BUG FIX: resize BEFORE masking
    hmap_224 = cv2.resize(hmap, (224, 224))

    border_mask = np.ones_like(hmap_224)
    border_mask[20:-20, 20:-20] = 0
    edge = np.sum(hmap_224 * border_mask) / (np.sum(hmap_224) + 1e-10)

    center_mask = np.zeros_like(hmap_224)
    center_mask[56:168, 56:168] = 1
    center = np.sum(hmap_224 * center_mask) / (np.sum(hmap_224) + 1e-10)

    class_heatmaps[true_cls].append(hmap)
    edge_scores[true_cls].append(edge)
    center_scores[true_cls].append(center)

print(f"\n{'Class':<12} {'N':<6} {'Edge-Bias':<12} {'Center-Focus':<14}")
print("-" * 50)
for c, name in enumerate(class_names):
    if class_heatmaps[c]:
        print(f"{name:<12} {len(class_heatmaps[c]):<6} {np.mean(edge_scores[c]):.4f}      {np.mean(center_scores[c]):.4f}")

# Figure 7: mean heatmaps
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for c, name in enumerate(class_names):
    if class_heatmaps[c]:
        mean_hmap = np.mean(class_heatmaps[c], axis=0)
        axes[c].imshow(mean_hmap, cmap='jet')
        axes[c].set_title(f'{name}\n(n={len(class_heatmaps[c])})')
        axes[c].axis('off')
plt.suptitle('Mean Grad-CAM Heatmaps — MobileNetV2 (7x7 native)', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/quantitative_gradcam_means.png', dpi=300, bbox_inches='tight')
plt.show()

# Figure 8: spatial bias bar chart
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
width = 0.35
ax.bar(x - width / 2, [np.mean(edge_scores[c]) for c in range(4)], width,
       label='Edge-Bias', color='#FF6B6B')
ax.bar(x + width / 2, [np.mean(center_scores[c]) for c in range(4)], width,
       label='Center-Focus', color='#4ECDC4')
ax.set_ylabel('Attention Fraction', fontsize=12)
ax.set_title('Grad-CAM Spatial Bias by Class — MobileNetV2', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/quantitative_gradcam_spatial_bias.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================= PART 5: EXTERNAL VALIDATION =========================

def eval_external(model, pre_fn, ext_path, ext_name):
    print(f"\n{'='*70}\nEXTERNAL: {ext_name}\n{'='*70}")
    if not ext_path or not os.path.exists(ext_path):
        print(f"  Not found: {ext_path}")
        return None
    try:
        ext_gen = ImageDataGenerator(preprocessing_function=pre_fn).flow_from_directory(
            ext_path, target_size=IMG_SIZE, batch_size=BATCH,
            class_mode='categorical', shuffle=False)
        ext_gen.reset()
        ext_prob = model.predict(ext_gen, verbose=0)
        ext_pred = np.argmax(ext_prob, axis=1)
        ext_true = ext_gen.classes
        ext_classes = list(ext_gen.class_indices.keys())
        overall = np.mean(ext_pred == ext_true)
        print(f"  Overall: {overall*100:.2f}% | Classes: {ext_classes}")
        for i, cls in enumerate(ext_classes):
            mask = ext_true == i
            acc = np.mean(ext_pred[mask] == i) if np.sum(mask) > 0 else 0
            print(f"    {cls:<15}: {acc*100:.2f}% (n={np.sum(mask)})")
        return overall
    except Exception as e:
        print(f"  ERROR: {e}")
        return None


print("\n" + "=" * 70)
print("PART 5: Cross-Dataset Generalization")
print("=" * 70)

mob_model = keras.models.load_model('/kaggle/working/mobilenetv2_seed42.keras')

ext_accs = {}
if FIGSHARE_DIR:
    ext_accs['Figshare'] = eval_external(mob_model, pre_mob, FIGSHARE_DIR, "Figshare")
if SARTAJ_TEST:
    ext_accs['SARTAJ'] = eval_external(mob_model, pre_mob, SARTAJ_TEST, "SARTAJ (Val set)")

# ========================= PART 6: ADVERSARIAL ROBUSTNESS (FGSM) =========================
print("\n" + "=" * 70)
print("PART 6: FGSM Adversarial Robustness Evaluation")
print("=" * 70)


def fgsm_attack(model, image, label, epsilon):
    image = tf.convert_to_tensor(image[np.newaxis, ...], dtype=tf.float32)
    label = tf.convert_to_tensor(label[np.newaxis, ...], dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image)
        prediction = model(image, training=False)
        loss = tf.keras.losses.categorical_crossentropy(label, prediction)
    gradient = tape.gradient(loss, image)
    signed_grad = tf.sign(gradient)
    adv_image = image + epsilon * signed_grad
    adv_image = tf.clip_by_value(adv_image, -1.0, 1.0)
    return adv_image


# Evaluate on a subset (first 200 images) for speed
robustness_results = {}
te_rob = ImageDataGenerator(preprocessing_function=pre_mob).flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=1, class_mode='categorical', shuffle=False)

eps_accs = {eps: [] for eps in EPSILONS}
sample_count = 0
max_samples = 200

for i in range(len(te_rob)):
    if sample_count >= max_samples:
        break
    x, y = te_rob[i]
    img = x[0]
    label = y[0]
    for eps in EPSILONS:
        if eps == 0.0:
            pred = mob_model.predict(img[np.newaxis, ...], verbose=0)
        else:
            adv = fgsm_attack(mob_model, img, label, eps)
            pred = mob_model.predict(adv, verbose=0)
        eps_accs[eps].append(int(np.argmax(pred) == np.argmax(label)))
    sample_count += 1

print(f"\nAdversarial Robustness (n={sample_count}):")
print(f"{'Epsilon':<10} {'Accuracy':<10} {'Retention':<12}")
print("-" * 35)
clean_acc = np.mean(eps_accs[0.0])
for eps in EPSILONS:
    acc = np.mean(eps_accs[eps])
    retention = acc / clean_acc if clean_acc > 0 else 0
    print(f"{eps:<10.2f} {acc*100:.2f}%    {retention*100:.1f}%")
    robustness_results[eps] = {'acc': acc, 'retention': retention}

# Plot robustness curve
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(EPSILONS, [np.mean(eps_accs[e])*100 for e in EPSILONS], 'o-', color='#DC2626', linewidth=2, markersize=8)
ax.set_xlabel('FGSM Epsilon', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Adversarial Robustness Curve — MobileNetV2', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/adversarial_robustness_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================= PART 7: CLINICAL READINESS INDEX (CRI) =========================
print("\n" + "=" * 70)
print("PART 7: Clinical Readiness Index (Novel Composite Metric)")
print("=" * 70)


def compute_cri(accuracy, ece, robustness_retention, external_acc, primary_acc):
    """
    CRI = 0.40*Accuracy + 0.20*Calibration + 0.20*Robustness + 0.20*Generalization
    All terms normalized to [0,1].
    """
    acc_term = accuracy
    cal_term = max(0.0, 1.0 - ece * 10)  # ECE=0.05 -> 0.5, scaled
    rob_term = robustness_retention
    gen_term = external_acc / primary_acc if primary_acc > 0 else 0.0
    cri = 0.40 * acc_term + 0.20 * cal_term + 0.20 * rob_term + 0.20 * gen_term
    return cri, acc_term, cal_term, rob_term, gen_term


print("\nPer-Model Clinical Readiness Index:")
print(f"{'Model':<18} {'CRI':<8} {'Acc':<8} {'Cal':<8} {'Rob':<8} {'Gen':<8}")
print("-" * 60)

# Use MobileNetV2 as the primary model for CRI (can repeat for others)
primary_acc = all_results['mobilenetv2']['acc']
primary_ece = all_results['mobilenetv2']['ece']
rob_ret = robustness_results[0.05]['retention']  # epsilon=0.05 retention

# Average external accuracy
avg_ext = np.mean([v for v in ext_accs.values() if v is not None]) if ext_accs else 0.0

for name, r in all_results.items():
    cri, a, c, ro, g = compute_cri(r['acc'], r['ece'], rob_ret, avg_ext, primary_acc)
    print(f"{name:<18} {cri:.4f}  {a:.4f}  {c:.4f}  {ro:.4f}  {g:.4f}")

# Ensemble CRI
ens_ece = ece_quick(y_true, cal_prob)
ens_cri, a, c, ro, g = compute_cri(cal_acc, ens_ece, rob_ret, avg_ext, primary_acc)
print(f"{'Ensemble (cal)':<18} {ens_cri:.4f}  {a:.4f}  {c:.4f}  {ro:.4f}  {g:.4f}")

# ========================= PART 8: CONFIDENCE ERROR ANALYSIS =========================
print("\n" + "=" * 70)
print("PART 8: High-Confidence Error Analysis")
print("=" * 70)

y_prob = np.load('/kaggle/working/mobilenetv2_seed42_y_prob.npy')
y_pred = np.load('/kaggle/working/mobilenetv2_seed42_y_pred.npy')
y_true = np.load('/kaggle/working/mobilenetv2_seed42_y_true.npy')

confidences = np.max(y_prob, axis=1)
correct_mask = (y_pred == y_true)
wrong_mask = ~correct_mask

print(f"Mean confidence (correct):   {np.mean(confidences[correct_mask]):.4f}")
print(f"Mean confidence (incorrect): {np.mean(confidences[wrong_mask]):.4f}")
print(f"Fraction of errors with confidence > 0.9: {np.mean(confidences[wrong_mask] > 0.9):.2%}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(confidences[correct_mask], bins=20, alpha=0.7, label='Correct',
        color='green', density=True)
ax.hist(confidences[wrong_mask], bins=20, alpha=0.7, label='Incorrect',
        color='red', density=True)
ax.axvline(x=0.9, color='black', linestyle='--', label='Confidence = 0.9')
ax.set_xlabel('Predicted Confidence', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Confidence Distribution: Correct vs Incorrect (MobileNetV2)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/confidence_error_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================= FINAL SUMMARY =========================
print("\n" + "=" * 70)
print("ALL DONE — FILES SAVED TO /kaggle/working/")
print("=" * 70)
print("""
Figures:
  1. reliability_diagram_all_models.png
  2. calibrated_ensemble_confidence.png
  3. quantitative_gradcam_means.png
  4. quantitative_gradcam_spatial_bias.png
  5. adversarial_robustness_curve.png
  6. confidence_error_analysis.png

Models:
  7. *_seed42.keras (backups)

Novel elements for your paper:
  - Focal Loss for class-imbalanced neuro-oncology data
  - Clinical Readiness Index (CRI) composite trust metric
  - FGSM adversarial robustness evaluation
  - Temperature-scaled ensemble with McNemar's significance test
  - Quantitative Grad-CAM spatial bias (edge vs center)
  - Dual external validation (Figshare + SARTAJ)
""")

[OK] Primary: /kaggle/input/datasets/ekrasafdar/brain-tumor-mri
     Train: /kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training
     Test:  /kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Testing
[OK] SARTAJ: /kaggle/input/datasets/ekrasafdar/sartajbhuvajibrain-tumor-classification-mri/Val
[OK] Figshare: /kaggle/input/datasets/ekrasafdar/sartajbhuvajibrain-tumor-classification-mri/Val

PART 1: Three Architectures + Calibration + Bootstrap CI

>>> MOBILENETV2 SEED 42

>>> Training mobilenetv2 (seed 42, focal_loss=True)...


/tmp/ipykernel_58/2239348412.py:118: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
I0000 00:00:1785758048.091451      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785758048.097494      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Epoch 1/30


2026-08-03 11:54:37.151965: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 11:54:37.289813: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1785758084.673806     153 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


140/140 ━━━━━━━━━━━━━━━━━━━━ 148s 829ms/step - accuracy: 0.7304 - loss: 0.1304 - val_accuracy: 0.7357 - val_loss: 0.0920 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 541ms/step - accuracy: 0.8567 - loss: 0.0543 - val_accuracy: 0.6991 - val_loss: 0.1535 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 545ms/step - accuracy: 0.8904 - loss: 0.0361 - val_accuracy: 0.7768 - val_loss: 0.1006 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 77s 552ms/step - accuracy: 0.9094 - loss: 0.0293 - val_accuracy: 0.8750 - val_loss: 0.0518 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 545ms/step - accuracy: 0.9286 - loss: 0.0184 - val_accuracy: 0.8929 - val_loss: 0.0488 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 541ms/step - accuracy: 0.9400 - loss: 0.0185 - val_accuracy: 0.9161 - val_loss: 0.0331 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 75s 538ms/step -

2026-08-03 13:10:14.005141: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 13:10:14.149294: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 13:10:14.500099: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 13:10:14.641788: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 13:10:14.784222: E external/local_xla/xla/stream_

140/140 ━━━━━━━━━━━━━━━━━━━━ 126s 630ms/step - accuracy: 0.6467 - loss: 0.1687 - val_accuracy: 0.7964 - val_loss: 0.0571 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 83s 591ms/step - accuracy: 0.8047 - loss: 0.0756 - val_accuracy: 0.8813 - val_loss: 0.0350 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 81s 576ms/step - accuracy: 0.8350 - loss: 0.0570 - val_accuracy: 0.9080 - val_loss: 0.0278 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 80s 574ms/step - accuracy: 0.8621 - loss: 0.0451 - val_accuracy: 0.9250 - val_loss: 0.0223 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 560ms/step - accuracy: 0.8806 - loss: 0.0346 - val_accuracy: 0.9214 - val_loss: 0.0237 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 79s 563ms/step - accuracy: 0.8864 - loss: 0.0311 - val_accuracy: 0.9384 - val_loss: 0.0182 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 79s 561ms/step -

In [ ]:
import numpy as np
from scipy.stats import binomtest
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

def load(name):
    return (np.load(f'/kaggle/working/{name}_seed42_y_prob.npy'),
            np.load(f'/kaggle/working/{name}_seed42_y_pred.npy'),
            np.load(f'/kaggle/working/{name}_seed42_y_true.npy'))

mob_prob, mob_pred, y_true = load('mobilenetv2')
res_prob, res_pred, _      = load('resnet50')

# 2-model ensemble
best_acc, best_w = 0, (0.5, 0.5)
for a in np.arange(0.1, 0.9, 0.05):
    b = 1 - a
    prob = a*mob_prob + b*res_prob
    acc = np.mean(np.argmax(prob, axis=1) == y_true)
    if acc > best_acc:
        best_acc = acc
        best_w = (a, b)

ens_prob = best_w[0]*mob_prob + best_w[1]*res_prob
ens_pred = np.argmax(ens_prob, axis=1)
ens_f1   = f1_score(y_true, ens_pred, average='weighted')

print(f"Weights: MNV2={best_w[0]:.2f}, RN50={best_w[1]:.2f}")
print(f"Ensemble: {best_acc*100:.2f}% | F1: {ens_f1*100:.2f}%")
print(f"MobileNetV2 alone: {np.mean(mob_pred==y_true)*100:.2f}%")
print(f"ResNet50 alone: {np.mean(res_pred==y_true)*100:.2f}%")